# 15 · Optimization

- **Level:** Intermediate → Advanced
- **Typical semester:** Semester 5–7 (Operations Research, Optimization Techniques, or ML electives)
- **Prerequisites:** 02, 06, 07
- **Used later in:** training every model. Each optimizer in a deep learning library is a formula from this file.


## 1. Linear programming

Standard form: maximize $c^\top x$ subject to $Ax\le b$, $x\ge0$. Optimum lies at a vertex of the feasible region.

**Simplex method:** move between adjacent vertices while the objective improves; Big-M and two-phase methods handle $\ge$ and $=$ constraints.

**Duality:** primal $\max c^\top x$, $Ax\le b$, $x\ge0$ ⟷ dual $\min b^\top y$, $A^\top y\ge c$, $y\ge0$. Weak duality $c^\top x\le b^\top y$; strong duality holds at the optimum. Complementary slackness: $y_i(b - Ax)_i = 0$.

Special LPs: transportation problem (north-west corner, least cost, Vogel's approximation; MODI for optimality), assignment problem (Hungarian method).


## 2. Convexity

Convex set: $\theta x + (1-\theta)y\in C$. Convex function: $f(\theta x + (1-\theta)y)\le\theta f(x) + (1-\theta)f(y)$.

First-order condition: $f(y)\ge f(x) + \nabla f(x)^\top(y - x)$. Second-order: $\nabla^2f\succeq0$.

**$\mu$-strongly convex:** $f(y)\ge f(x) + \nabla f^\top(y-x) + \frac\mu2\|y - x\|^2$. **$L$-smooth:** $\|\nabla f(x) - \nabla f(y)\|\le L\|x - y\|$. Condition number $\kappa = L/\mu$.

For convex $f$, every local minimum is global. Losses like MSE, logistic and hinge are convex in a linear model's weights; neural network losses are not.


## 3. Unconstrained optimality

Necessary: $\nabla f(x^*) = 0$. Sufficient: also $\nabla^2f(x^*)\succ0$. Saddle point: Hessian has both positive and negative eigenvalues (very common in high-dimensional deep learning).


## 4. Constrained optimization: Lagrange and KKT

Problem: $\min f(x)$ s.t. $g_i(x)\le0$, $h_j(x) = 0$. Lagrangian $\mathcal L = f + \sum\lambda_ig_i + \sum\nu_jh_j$.

**KKT conditions:**
1. stationarity $\nabla f + \sum\lambda_i\nabla g_i + \sum\nu_j\nabla h_j = 0$
2. primal feasibility $g_i\le0$, $h_j = 0$
3. dual feasibility $\lambda_i\ge0$
4. complementary slackness $\lambda_ig_i = 0$

Dual function $q(\lambda,\nu) = \inf_x\mathcal L$; weak duality $q\le f^*$; strong duality under Slater's condition (convex problem with a strictly feasible point).

**ML note:** the SVM dual and the "only support vectors have $\alpha_i>0$" result come straight from KKT.


## 5. Gradient descent and its convergence

$$\theta_{t+1} = \theta_t - \eta\nabla f(\theta_t)$$

For $L$-smooth convex $f$ with $\eta = 1/L$: $f(\theta_t) - f^*\le\frac{L\|\theta_0 - \theta^*\|^2}{2t}$ (rate $O(1/t)$).
For $\mu$-strongly convex: $\|\theta_t - \theta^*\|^2\le(1 - \mu/L)^t\|\theta_0 - \theta^*\|^2$ (linear rate).

Line search: exact, or backtracking with the Armijo condition $f(x - \eta\nabla f)\le f(x) - c\eta\|\nabla f\|^2$.

**Projected gradient:** $\theta\leftarrow\Pi_C(\theta - \eta\nabla f)$. **Proximal gradient:** $\theta\leftarrow\text{prox}_{\eta g}(\theta - \eta\nabla f)$; for L1, prox is soft-thresholding $\text{sign}(x)\max(|x| - \eta\lambda, 0)$ (ISTA; accelerated version FISTA).

**Coordinate descent:** optimize one coordinate at a time (used for Lasso).


## 6. Second-order and quasi-Newton methods

**Newton:** $\theta_{t+1} = \theta_t - [\nabla^2f]^{-1}\nabla f$ (quadratic convergence near the optimum)

**Gauss-Newton** (least squares $\frac12\|r(\theta)\|^2$): $\Delta = -(J^\top J)^{-1}J^\top r$. **Levenberg-Marquardt:** $\Delta = -(J^\top J + \lambda I)^{-1}J^\top r$

**BFGS** Hessian update with $s = \theta_{t+1} - \theta_t$, $y = \nabla f_{t+1} - \nabla f_t$:
$$B_{t+1} = B_t + \frac{yy^\top}{y^\top s} - \frac{B_tss^\top B_t}{s^\top B_ts}$$
**L-BFGS** stores only the last $m$ pairs $(s, y)$.

**Natural gradient:** $\theta\leftarrow\theta - \eta F^{-1}\nabla L$ with Fisher matrix $F$; approximated by K-FAC ($F\approx A\otimes G$) and Shampoo.

**Conjugate gradient** (see 10 §3) for large quadratic problems and Hessian-free optimization.


## 7. Stochastic gradient methods

Mini-batch gradient $g_t = \frac1B\sum_{i\in\mathcal B}\nabla\ell_i(\theta_t)$, an unbiased estimate of $\nabla f$.

**SGD:** $\theta_{t+1} = \theta_t - \eta_tg_t$; convex rate $O(1/\sqrt t)$, strongly convex $O(1/t)$ with $\eta_t\propto1/t$.

**Momentum (heavy ball):** $v_{t+1} = \beta v_t + g_t$, $\;\theta_{t+1} = \theta_t - \eta v_{t+1}$

**Nesterov:** gradient evaluated at the look-ahead point $\theta_t - \eta\beta v_t$; optimal rate $O(1/t^2)$ for smooth convex problems.

**Adagrad:** $G_t = G_{t-1} + g_t^2$, $\;\theta\leftarrow\theta - \frac{\eta}{\sqrt{G_t} + \epsilon}g_t$

**RMSProp:** $v_t = \rho v_{t-1} + (1-\rho)g_t^2$, $\;\theta\leftarrow\theta - \frac\eta{\sqrt{v_t}+\epsilon}g_t$

**Adam:**
$$m_t = \beta_1m_{t-1} + (1-\beta_1)g_t, \qquad v_t = \beta_2v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^t}, \qquad \theta_t = \theta_{t-1} - \eta\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}$$
Defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

**AdamW** (decoupled weight decay): $\theta_t = \theta_{t-1} - \eta\left(\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon} + \lambda\theta_{t-1}\right)$

**Lion:** $c_t = \beta_1m_{t-1} + (1-\beta_1)g_t$, $\;\theta\leftarrow\theta - \eta(\text{sign}(c_t) + \lambda\theta)$, $\;m_t = \beta_2m_{t-1} + (1-\beta_2)g_t$

**Muon** (for 2D weight matrices): orthogonalize the momentum $M$ before the step, $\theta\leftarrow\theta - \eta\,\text{Orth}(M)$, $\text{Orth}(M) = UV^\top$ for $M = U\Sigma V^\top$ (computed with a Newton-Schulz iteration).

Others: AMSGrad ($\hat v_t = \max(\hat v_{t-1}, v_t)$), RAdam, NAdam, Adafactor (factored second moments), LAMB/LARS (layer-wise trust ratio $\frac{\|\theta\|}{\|\text{update}\|}$), Sophia (clipped diagonal Hessian), Shampoo ($L^{-1/4}GR^{-1/4}$).

**Variance reduction:** SVRG $g = \nabla\ell_i(\theta) - \nabla\ell_i(\tilde\theta) + \nabla f(\tilde\theta)$; SAG, SAGA.


## 8. Learning-rate schedules and training tricks

Step decay $\eta_t = \eta_0\gamma^{\lfloor t/s\rfloor}$; exponential $\eta_0e^{-kt}$; inverse-sqrt $\eta_0/\sqrt t$ (original Transformer)

**Cosine annealing:** $\eta_t = \eta_{\min} + \frac12(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{\pi t}{T}\right)$

Linear warmup $\eta_t = \eta_{\max}\frac tT_w$; one-cycle; warmup-stable-decay (WSD)

**Gradient clipping by norm:** $g\leftarrow g\cdot\min\left(1, \frac{c}{\|g\|}\right)$

**Polyak / exponential moving average of weights:** $\bar\theta_t = \alpha\bar\theta_{t-1} + (1-\alpha)\theta_t$

**Linear scaling rule:** when multiplying batch size by $k$, multiply the learning rate by $k$ (for SGD; roughly $\sqrt k$ for Adam).


## 9. Derivative-free and global optimization

Nelder-Mead simplex, golden-section search (interval ratio $0.618$), simulated annealing (accept worse moves with probability $e^{-\Delta/T}$), genetic algorithms, CMA-ES, particle swarm ($v\leftarrow wv + c_1r_1(p - x) + c_2r_2(g - x)$)

**Bayesian optimization:** Gaussian-process surrogate + acquisition function. Expected improvement $EI(x) = (\mu - f^* - \xi)\Phi(Z) + \sigma\phi(Z)$, $Z = \frac{\mu - f^* - \xi}{\sigma}$; UCB $= \mu + \kappa\sigma$.


## 10. Min-max, games and bilevel problems

Saddle-point problem $\min_x\max_yf(x,y)$ (GANs, adversarial training, robust optimization). Gradient descent-ascent can cycle; extragradient and optimistic GD fix this for convex-concave problems.

**Von Neumann minimax theorem:** for convex-concave $f$ on compact convex sets, $\min_x\max_yf = \max_y\min_xf$.

**Bilevel optimization:** $\min_\lambda\mathcal L_{val}(\theta^*(\lambda))$ s.t. $\theta^*(\lambda) = \arg\min_\theta\mathcal L_{train}(\theta,\lambda)$ (hyperparameter optimization, meta-learning, MAML). Implicit gradient: $\frac{d\theta^*}{d\lambda} = -[\nabla^2_\theta\mathcal L]^{-1}\nabla^2_{\theta\lambda}\mathcal L$.


## 11. Dynamic programming

**Bellman's principle of optimality:** an optimal policy has optimal sub-policies. $V(s) = \max_a[r(s,a) + V(s')]$.

Used in shortest paths, sequence alignment, the Viterbi algorithm, CTC loss and RL (see 21).


## 12. Optimal transport

$\min_{\gamma\in\Pi(a,b)}\langle C,\gamma\rangle$. **Entropic regularization + Sinkhorn:** $\gamma = \text{diag}(u)K\text{diag}(v)$, $K = e^{-C/\varepsilon}$, alternate $u\leftarrow a/(Kv)$, $v\leftarrow b/(K^\top u)$.
